# OMF export — GSWA sample data

Round-trip a subset of the GSWA sample data through Open Mining Format (OMF v1), the MIT-licensed mining interchange format read/written by Leapfrog, 3DEXPERIENCE, Deswik and Micromine.

Requires the optional extra:

```bash
pip install baselode[omf]
```

In [ ]:
import baselode.drill.data as drill
import baselode.drill.desurvey as desurvey
import baselode.drill.omf as omf_io

DATA = '../test/data/gswa'
collars = drill.load_collars(f'{DATA}/gswa_sample_collars.csv')
surveys = drill.load_surveys(f'{DATA}/gswa_sample_survey.csv')
assays = drill.load_assays(f'{DATA}/gswa_sample_assays.csv')

# Pick the first 20 holes with at least 2 survey stations
counts = surveys['hole_id'].value_counts()
sample_holes = counts[counts >= 2].head(20).index.tolist()
collars_sub = collars[collars['hole_id'].isin(sample_holes)]
surveys_sub = surveys[surveys['hole_id'].isin(sample_holes)]
assays_sub = assays[assays['hole_id'].isin(sample_holes)]

len(collars_sub), len(surveys_sub), len(assays_sub)

## Desurvey the holes

OMF needs projected `(easting, northing, elevation)` coords.  Desurvey produces them as a side effect.

In [ ]:
traces = desurvey.minimum_curvature_desurvey(collars_sub, surveys_sub, step=5.0)
traces.head()

## Build the OMF elements

GSWA collars only carry lat/lon, so derive collar positions from the head of each trace.  In a real project you'd typically project the collar table to a working CRS first via `baselode.extent.Extent.to_crs`.

In [ ]:
collar_xyz = (
    traces.sort_values('md')
          .groupby('hole_id')
          .first()
          .reset_index()
          [['hole_id', 'easting', 'northing', 'elevation']]
          .merge(collars_sub[['hole_id', 'maxdepth']].rename(columns={'maxdepth': 'max_depth'}), on='hole_id')
)

collar_el = omf_io.collars_to_omf_points(collar_xyz, attribute_cols=['max_depth'])
trace_el = omf_io.traces_to_omf_lines(traces)

# Pick a handful of assay columns to attach as per-segment data
assay_value_cols = [c for c in assays_sub.columns
                     if c not in ('hole_id','from','to','mid','datasource_hole_id','sample_id','extra')][:4]
assay_el = omf_io.intervals_to_omf_lines(
    assays_sub, traces, name='assay', value_cols=assay_value_cols,
)

{
    'collar vertices': collar_el.geometry.vertices.array.shape,
    'trace vertices':  trace_el.geometry.vertices.array.shape,
    'trace segments':  trace_el.geometry.segments.array.shape,
    'assay segments':  assay_el.geometry.segments.array.shape,
}

## Write and read back

In [ ]:
from pathlib import Path

path = Path('/tmp/gswa-subset.omf')
omf_io.write_omf(
    [collar_el, trace_el, assay_el],
    path,
    name='gswa-subset',
    author='baselode',
    description='GSWA 20-hole subset',
)
print(f'{path}: {path.stat().st_size / 1024:.1f} KB')

project = omf_io.read_omf(path)
[(el.name, type(el).__name__) for el in project.elements]